In [17]:
import os
import torch
import cv2
import glob
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torchvision.datasets import CocoDetection
from pycocotools.coco import COCO
import multiprocessing

In [20]:


##################################################
# Utility: Create mapping from COCO category IDs to contiguous indices
##################################################
def create_id_mapping(annFile):
    coco = COCO(annFile)
    cats = coco.loadCats(coco.getCatIds())
    # Sorted list of category ids (e.g., [1, 2, 3, ...])
    cat_ids = sorted([cat['id'] for cat in cats])
    # Map original COCO category id to new index (0, 1, 2, ...)
    id_to_idx = {cat_id: idx for idx, cat_id in enumerate(cat_ids)}
    return id_to_idx

##################################################
# Custom Dataset: Wrap CocoDetection to extract a single label per image
##################################################
class CocoSingleLabelDataset(Dataset):
    def __init__(self, root, annFile, transform=None, id_to_idx=None):
        """
        Args:
            root (str): Directory with all the images (e.g., train2017).
            annFile (str): Path to json annotations (e.g., instances_train2017.json).
            transform: Transformations to apply to the images.
            id_to_idx (dict): Mapping from original COCO category id to new index.
        """
        self.coco = CocoDetection(root=root, annFile=annFile, transform=transform)
        self.id_to_idx = id_to_idx
        # Filter out images without any annotations
        self.indices = [i for i in range(len(self.coco)) if len(self.coco[i][1]) > 0]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        print("Entered Get item function")
        real_idx = self.indices[idx]
        img, anns = self.coco[real_idx]
        # For classification, we take the first annotation's category id.
        label = anns[0]['category_id']
        if self.id_to_idx:
            label = self.id_to_idx[label]
        print("Exiting Get item function")
        return img, label

##################################################
# Define a Simple CNN Model for Classification
##################################################
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=80):
        """
        A simple CNN with several convolutional layers followed by fully-connected layers.
        Adjust num_classes to match the number of classes in your dataset.
        """
        print("Entered CNN function")
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 224 -> 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 112 -> 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 56 -> 28
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x

##################################################
# Training Function
##################################################
def train_model(model, dataloader, criterion, optimizer, device, num_epochs=5):
    model.train()
    print("Entered train function")
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        
        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

In [ ]:
##################################################
# Main Execution
##################################################
if __name__ == "__main__":
    # ======================================
    # Downloading & Setting Up the COCO Dataset
    # ======================================
    # You can download the COCO 2017 Train images and annotations as follows:
    #
    # 1. Train images (JPEGs):
    #    URL: http://images.cocodataset.org/zips/train2017.zip
    #    Download and extract to a folder (e.g., /path/to/coco/train2017)
    #
    # 2. Annotations:
    #    URL: http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    #    Download and extract; the file of interest is instances_train2017.json,
    #    e.g., /path/to/coco/annotations/instances_train2017.json
    #
    # Update the paths below to where you extracted these files.
    
    coco_root = "/Users/shivansh052k/Documents/UB/Academics/Semester_2/CV & IP/Project/Dataset/archive/train2017"  # e.g., "/home/username/coco/train2017"
    annFile = "/Users/shivansh052k/Documents/UB/Academics/Semester_2/CV & IP/Project/Dataset/archive/annotations/instances_train2017.json"  # e.g., "/home/username/coco/annotations/instances_train2017.json"
    
    # Create a mapping for category ids (COCO has 80 classes)
    id_to_idx = create_id_mapping(annFile)
    
    # ======================================
    # Data Transforms & DataLoader Setup
    # ======================================
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    print("Entering Dataset Class")
    dataset = CocoSingleLabelDataset(root=coco_root, annFile=annFile, transform=transform, id_to_idx=id_to_idx)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)
    
    # ======================================
    # Initialize Model, Loss, and Optimizer
    # ======================================
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(id_to_idx)  # Typically 80 for COCO
    model = SimpleCNN(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    
    # ======================================
    # Train the Model
    # ======================================
    print("Entering Train function")
    train_model(model, dataloader, criterion, optimizer, device, num_epochs=1)

loading annotations into memory...
Done (t=0.35s)
creating index...
index created!
loading annotations into memory...
Done (t=0.28s)
creating index...
index created!


KeyboardInterrupt: 